# 🚀 GDGoC-HCMUS AI Challenge 2026 – BomIT Training

**Chạy trên Kaggle GPU (Tesla T4 miễn phí)**

Các bước:
1. Clone repo từ GitHub
2. Cài dependencies
3. Train agent với PPO + Curriculum
4. Download checkpoint về máy

In [ ]:
# ── Kiểm tra GPU ──────────────────────────────────────────────────────────
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Clone repo từ GitHub ──────────────────────────────────────────────────
# ⚠️  Thay YOUR_USERNAME/YOUR_REPO_NAME bằng repo thật của bạn
GITHUB_REPO = 'YOUR_USERNAME/YOUR_REPO_NAME'

import os
REPO_NAME = GITHUB_REPO.split('/')[-1]
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if os.path.exists(WORK_DIR):
    print('Repo đã tồn tại, pull update...')
    os.system(f'cd {WORK_DIR} && git pull')
else:
    print('Đang clone repo...')
    os.system(f'git clone https://github.com/{GITHUB_REPO} {WORK_DIR}')

os.chdir(WORK_DIR)
print(f'\n✅ Working dir: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ── Cài dependencies ──────────────────────────────────────────────────────
# Kaggle đã có torch/numpy, chỉ cài thêm nếu cần engine BomIT

# Nếu BTC cung cấp engine dạng pip package:
# !pip install bomit-engine -q

# Nếu engine là file .whl đã upload vào Kaggle dataset:
# !pip install /kaggle/input/bomit-engine/bomit_engine-*.whl -q

# Kiểm tra import
import sys
sys.path.insert(0, WORK_DIR)

import torch, numpy as np
from utils.state_encoder import encode_state
from agent.network import BomberNet
from training.ppo_trainer import PPOTrainer, PPOConfig
print('✅ Tất cả import OK')

In [ ]:
# ── Setup môi trường ──────────────────────────────────────────────────────
# Thay đoạn này khi có engine BomIT thật từ BTC

def make_env():
    """
    Option A: Dùng engine BomIT thật
        from bomit_engine import BomITEnv
        return BomITEnv(grid_size=13, n_agents=4)

    Option B: Mock env (chạy thử pipeline)
    """
    from train import MockBomITEnv
    return MockBomITEnv()

# Test env
env = make_env()
obs = env.reset()
print(f'✅ Env reset OK | Agents: {list(obs.keys())}')
env.close()

In [ ]:
# ── Cấu hình training ─────────────────────────────────────────────────────
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES  = 5000    # tăng lên 10000-20000 nếu có đủ thời gian GPU
SAVE_DIR  = '/kaggle/working/checkpoints'

cfg = PPOConfig(
    lr            = 3e-4,
    n_steps       = 512,
    batch_size    = 128,   # GPU cho phép batch lớn hơn
    n_epochs      = 4,
    clip_eps      = 0.2,
    entropy_coef  = 0.01,
    save_dir      = SAVE_DIR,
    log_freq      = 50,
    pool_size     = 10,
)

print(f'Device   : {DEVICE}')
print(f'Episodes : {EPISODES}')
print(f'Save dir : {SAVE_DIR}')

In [ ]:
# ── Training chính ────────────────────────────────────────────────────────
import os
os.makedirs(SAVE_DIR, exist_ok=True)

trainer = PPOTrainer(
    env_factory = make_env,
    cfg         = cfg,
    agent_id    = 0,
    device      = DEVICE,
)

# Nếu muốn tiếp tục từ checkpoint cũ:
# trainer.load('/kaggle/working/checkpoints/agent_2000.pt')

trainer.train(total_episodes=EPISODES)

In [ ]:
# ── Kiểm tra kết quả ──────────────────────────────────────────────────────
import os
checkpoints = sorted(os.listdir(SAVE_DIR))
print('Checkpoints đã lưu:')
for f in checkpoints:
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / 1e6
    print(f'  {f}  ({size:.1f} MB)')

# Test agent cuối
from agent.agent import BomITAgent
agent = BomITAgent(
    model_path = f'{SAVE_DIR}/agent_final.pt',
    agent_id   = 0,
    device     = DEVICE,
)
import numpy as np
fake_obs = {
    'board': np.zeros((13,13), dtype=int), 'bombs': [], 'flames': [], 'items': [],
    'agents': [{'position': (1,1), 'alive': True, 'bomb_count': 1,
                'blast_strength': 2, 'can_kick': 0}] * 4,
}
action = agent.act(fake_obs)
names  = ['STOP','UP','DOWN','LEFT','RIGHT','BOMB']
print(f'\n✅ Agent test OK → action = {action} ({names[action]})')

In [ ]:
# ── Nén và download checkpoint ────────────────────────────────────────────
import shutil

ZIP_PATH = '/kaggle/working/bomit_checkpoints.zip'
shutil.make_archive('/kaggle/working/bomit_checkpoints', 'zip', SAVE_DIR)
size = os.path.getsize(ZIP_PATH) / 1e6
print(f'✅ Đã tạo: {ZIP_PATH}  ({size:.1f} MB)')
print()
print('👉 Download file từ: Kaggle → Output tab bên phải → bomit_checkpoints.zip')
print('👉 Giải nén vào thư mục checkpoints/ trong repo local')
print('👉 Tiếp tục train: python train.py --resume checkpoints/agent_final.pt')

## 💡 Tips khi dùng Kaggle GPU

| Mẹo | Chi tiết |
|-----|----------|
| **Tiếp tục train** | Upload checkpoint cũ vào Kaggle Dataset, load lại bằng `trainer.load(path)` |
| **Tăng tốc** | Tăng `batch_size=256`, `n_steps=1024` khi dùng GPU T4 |
| **Tránh hết thời gian** | Lưu checkpoint mỗi 500 ep, Kaggle giới hạn ~9h/session |
| **Nhiều session** | Dùng `trainer.load()` để tiếp tục qua nhiều session khác nhau |
| **Theo dõi** | Xem log cell Output để biết win rate và phase curriculum |